# APEX Triton validation on two NVIDIA T4 GPUs

Runs correctness tests for the fused Mamba-3 recurrent scan (including gradients) and fused decode step, then measures them against eager PyTorch. It also trains the HOP-MIX + LRCM + Mamba-3 hybrid with data parallelism across both GPUs and runs autoregressive generation on each GPU. Timings are measured on this Kaggle runtime; no speedup is assumed in advance.

In [ ]:
import subprocess
import sys
import torch

assert torch.cuda.is_available(), 'Kaggle CUDA GPU is required'
assert torch.cuda.device_count() == 2, f'Expected exactly two GPUs, got {torch.cuda.device_count()}'
gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
assert all('T4' in name for name in gpu_names), f'Expected two T4 GPUs, got {gpu_names}'
print('GPUs:', gpu_names)
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/APEX')
if repo.exists():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=repo, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/bueormnew/APEX.git', str(repo)], check=True)
sys.path.insert(0, str(repo))
import triton
print('Repository:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=repo, text=True).strip())
print('Triton:', triton.__version__)
assert triton.__version__

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_triton_kernels.py'], cwd=repo, check=True)

In [ ]:
subprocess.run([sys.executable, 'kaggle/run_benchmarks.py'], cwd=repo, check=True)
report = Path('/kaggle/working/apex_triton_benchmark.json')
assert report.is_file() and report.stat().st_size > 0
print('Benchmark artifact:', report)